<a href="https://colab.research.google.com/github/Andru-1987/data_science_ii_96085/blob/main/08_semana/clase_practica/Analisis_Mercado_Inmobiliario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Análisis Univariado y Bivariado — Caso Práctico: Mercado Inmobiliario**
Este notebook interactivo implementa un caso práctico completo de análisis univariado y bivariado aplicado a un dataset sintético del mercado inmobiliario de 600 propiedades.

## **1. Configuración del Entorno y Generación del Dataset**
En esta sección importamos las librerías necesarias (Pandas, NumPy, SciPy y Plotly) y ejecutamos el generador del dataset sintético con relaciones matemáticas conocidas y controladas.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go

# Configuración de reproducibilidad
rng = np.random.default_rng(42)
n = 600

barrios = np.array(["Norte", "Centro", "Sur", "Costa"])
barrio_mult = {"Norte": 1.25, "Centro": 1.0, "Sur": 0.75, "Costa": 1.6}
barrio = rng.choice(barrios, size=n, p=[0.25, 0.35, 0.25, 0.15])

tipos = np.array(["Casa", "Departamento", "PH"])
tipo = rng.choice(tipos, size=n, p=[0.35, 0.45, 0.20])
tipo_mult = {"Casa": 1.15, "Departamento": 1.0, "PH": 0.9}

# Metros cuadrados: lognormal para generar asimetría positiva realista
m2 = rng.lognormal(mean=4.3, sigma=0.35, size=n)
m2 = np.clip(m2, 25, None)

# Antigüedad: relación NO lineal (en U, no monótona) con el precio
antiguedad = rng.integers(0, 60, size=n)
efecto_antiguedad_frac = 0.00035 * (antiguedad - 25) ** 2

ambientes = np.clip(np.round(m2 / 28 + rng.normal(0, 0.4, n)), 1, 6)

precio_base = 900 * m2
ruido_precio = rng.lognormal(mean=0, sigma=0.15, size=n)

precio_usd = np.array([
    precio_base[i] * barrio_mult[barrio[i]] * tipo_mult[tipo[i]]
    * (1 + efecto_antiguedad_frac[i]) * ruido_precio[i]
    for i in range(n)
])

# Inyección de outliers de lujo
idx_lujo = rng.choice(n, size=12, replace=False)
precio_usd[idx_lujo] *= rng.uniform(2.2, 3.5, size=12)

# Impuestos anuales fuertemente correlacionados con el precio (multicolinealidad)
impuestos_anuales_usd = precio_usd * 0.012 * (1 + rng.normal(0, 0.10, n))
impuestos_anuales_usd = np.clip(impuestos_anuales_usd, 200, None)

dias_en_mercado = rng.poisson(lam=45, size=n) + (antiguedad // 10)

df = pd.DataFrame({
    "id": range(1, n + 1),
    "barrio": barrio,
    "tipo_propiedad": tipo,
    "metros_cuadrados": m2.round(1),
    "ambientes": ambientes.astype(int),
    "antiguedad_anios": antiguedad,
    "precio_usd": precio_usd.round(0),
    "impuestos_anuales_usd": impuestos_anuales_usd.round(0),
    "dias_en_mercado": dias_en_mercado,
})

print("Dimensiones del dataset:", df.shape)
df.head(8)

## **2. Análisis Univariado**
### **2.1 Variable Numérica: precio_usd**
Calculamos los principales estadísticos descriptivos, aplicamos la regla del IQR para la detección de outliers y visualizamos la asimetría de la distribución.

In [ ]:
media = df["precio_usd"].mean()
mediana = df["precio_usd"].median()
std = df["precio_usd"].std()

q1 = df["precio_usd"].quantile(0.25)
q3 = df["precio_usd"].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr
mask_outliers_sup = df["precio_usd"] > limite_superior
n_outliers = mask_outliers_sup.sum()
pct_outliers = (n_outliers / len(df["precio_usd"].dropna())) * 100

skewness = df["precio_usd"].skew()

resumen = pd.DataFrame({
    "Métrica": [
        "Media", "Mediana", "Desv. estándar", "Q1", "Q3", "IQR",
        "Límite superior (Q3 + 1.5·IQR)", "Outliers detectados", "Skewness"
    ],
    "Valor": [
        f"USD {media:,.3f}", f"USD {mediana:,.3f}", f"USD {std:,.3f}",
        f"{q1:,.3f}", f"{q3:,.3f}", f"{iqr:,.3f}",
        f"{limite_superior:,.3f}", f"{n_outliers} ({pct_outliers:.1f}%)",
        f"{skewness:.2f}"
    ]
})
display(resumen)

# Gráfico interactivo: Histograma y Boxplot marginal
fig_precio = px.histogram(
    df, x="precio_usd", marginal="box", nbins=50,
    title="<b>Distribución de Precios: Asimetría y Outliers de Lujo</b><br><sup>La media es arrastrada hacia la derecha por los valores atípicos.</sup>",
    labels={"precio_usd": "Precio (USD)"},
    color_discrete_sequence=["#1f77b4"],
    template="plotly_white"
)
fig_precio.add_vline(x=media, line_dash="dash", line_color="red", annotation_text="Media")
fig_precio.add_vline(x=mediana, line_dash="dash", line_color="green", annotation_text="Mediana")
fig_precio.show()


**Preguntas guía → Hallazgos:**

- *¿La media y la mediana coinciden?* No — la media (90.347) es **~12% mayor** que la mediana (80.379). Esto ya nos avisa, sin graficar nada, que la distribución no es simétrica.
- *¿Hay outliers?* Sí, 23 propiedades (3.8%) superan el límite de Q3+1.5·IQR. Corresponden en su mayoría a las propiedades "de lujo" que inyectamos deliberadamente.
- *¿Qué forma tiene la distribución?* Un histograma mostraría una cola larga a la derecha (precios muy altos poco frecuentes) — coherente con el hallazgo de skewness que calculamos.


### **2.2 Variable Categórica: tipo_propiedad**

In [ ]:
conteo_tipo = df['tipo_propiedad'].value_counts().reset_index()
conteo_tipo.columns = ['Tipo de Propiedad', 'Cantidad']
conteo_tipo['Porcentaje (%)'] = (conteo_tipo['Cantidad'] / len(df)) * 100
display(conteo_tipo)

**Hallazgo:** el mercado está dominado por departamentos; los PH son el segmento minoritario — dato relevante antes de comparar precios por tipo, porque ese grupo tendrá menos poder estadístico.


In [ ]:
fig_tipo = px.bar(
    conteo_tipo, x='Tipo de Propiedad', y='Cantidad', text='Cantidad',
    title="<b>Composición del Mercado Inmobiliario</b>",
    color='Tipo de Propiedad',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    template="plotly_white"
)
fig_tipo.update_traces(textposition='outside')
fig_tipo.show()


In [ ]:
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Ejecutar la prueba de Tukey
# 1° argumento: la variable numérica (precios)
# 2° argumento: la variable categórica (grupos)
# alpha=0.05 es el nivel de significancia estándar (95% de confianza)
tukey = pairwise_tukeyhsd(endog=df['precio_usd'],
                          groups=df['tipo_propiedad'],
                          alpha=0.05)

print(tukey)


**Tukey**
La interpretacion :

Vamos a interpretarlos de forma sencilla mirando la columna clave: reject (que te dice si rechazamos la idea de que son iguales) y meandiff (la diferencia de precios).

Aquí está la conclusión de tus datos:
## 1. Casa vs. Departamento (reject: True)

* Resultado: Sí hay una diferencia significativa en el precio promedio.
* Interpretación: La diferencia (meandiff) es de -$12,863.31. Como el número es negativo, significa que los Departamentos son, en promedio, unos 12,863 dólares más baratos que las Casas (o visto al revés, las Casas son significativamente más caras). El p-valor (p-adj: 0.0193) es menor a 0.05, lo que valida estadísticamente esta brecha.

## 2. Casa vs. PH (reject: True)

* Resultado: Sí hay una diferencia significativa en el precio promedio.
* Interpretación: La diferencia es de -$15,843.80. Esto nos indica que los PHs son, en promedio, unos 15,843 dólares más baratos que las Casas. Al igual que el caso anterior, el p-valor (0.0174) confirma que no es una diferencia por casualidad.

## 3. Departamento vs. PH (reject: False)

* Resultado: No hay una diferencia significativa.
* Interpretación: Aunque en tu muestra el promedio de los PHs dió unos $2,980 dólares menos que los departamentos (meandiff: -2980.48), estadísticamente se consideran similares. El p-valor es muy alto (p-adj: 0.8559), lo que significa que la dispersión de precios dentro de ambos grupos es tan grande que no podemos asegurar que un tipo sea realmente más caro que el otro en el mercado general.

------------------------------
## Resumen del Mercado
Tus datos demuestran que el mercado se divide en dos escalones claros:

   1. Las Casas están en la cima (son el grupo significativamente más caro).
   2. Los Departamentos y los PHs comparten el escalón inferior, teniendo precios promedio muy similares entre sí desde el punto de vista estadístico.


## **3. Análisis Bivariado**
### **3.1 Numérica vs Numérica: metros_cuadrados vs precio_usd**

In [ ]:
r, p_val = stats.pearsonr(df["metros_cuadrados"], df["precio_usd"])
print(f"Coeficiente de Correlación de Pearson (r): {r:.3f} (p-valor: {p_val:.4e})")

fig_m2_precio = px.scatter(
    df, x="metros_cuadrados", y="precio_usd", color="tipo_propiedad",
    trendline="ols", opacity=0.7,
    title=f"<b>Tamaño vs Precio (r = {r:.3f})</b>",
    labels={"metros_cuadrados": "Metros Cuadrados", "precio_usd": "Precio (USD)"},
    template="plotly_white"
)
fig_m2_precio.show()


**Resultado ANOVA:** F = 5.12, p = 0.006

> *El ANOVA (análisis de varianza) es una técnica estadística que sirve para comparar las medias de tres o más grupos y saber si existen diferencias significativas entre ellos*

ANOVA no mira solo los promedios; mira la varianza (la dispersión). Compara qué tan separados están los promedios de los grupos entre sí (variabilidad entre grupos) contra qué tan dispersos están los precios dentro de cada misma bolsa (variabilidad dentro de los grupos).

Si la distancia entre los grupos es mucho mayor que la dispersión interna, ANOVA te dice: "Sí, el tipo de propiedad influye en el precio (p-valor < 0.05)".

El problema de ANOVA: Es una prueba global. Te dice que hay una diferencia, pero no te dice dónde. No sabés si las Casas son más caras que los Departamentos, si los PHs se diferencian de las Casas, o si los tres son completamente distintos entre sí. Para resolver esto, usamos la Prueba de Tukey.


**Pregunta → Hallazgo:** *¿El tipo de propiedad influye en el precio?* Sí, la diferencia entre grupos es estadísticamente significativa (p<0.05): las casas son en promedio ~19% más caras que los PH. Un boxplot agrupado confirmaría visualmente que la distribución de "Casa" está desplazada hacia arriba.

**Tukey  → La Prueba Post-Hoc de Tukey (HSD)**
La prueba de Tukey (Honestly Significant Difference) compara todos los pares posibles de grupos uno por uno (Casas vs. Deptos, Casas vs. PHs, Deptos vs. PHs) y corrige el margen de error estadístico para que no cometamos falsos positivos al hacer múltiples comparaciones.


### **3.2 Categórica vs Numérica: tipo_propiedad vs precio_usd (ANOVA y Tukey HSD)**

In [ ]:
# Análisis ANOVA
grupos = [g["precio_usd"].values for _, g in df.groupby("tipo_propiedad")]
f_stat, p_anova = stats.f_oneway(*grupos)
print(f"ANOVA F-stat: {f_stat:.2f}, p-valor: {p_anova:.4e}")

# Estadísticos por grupo
stats_tipo = df.groupby("tipo_propiedad")["precio_usd"].agg(
    Media="mean", Mediana="median", Std="std"
).reset_index()
display(stats_tipo)

fig_anova = px.box(
    df, x="tipo_propiedad", y="precio_usd", color="tipo_propiedad",
    title="<b>Precios según Tipo de Propiedad (ANOVA)</b>",
    labels={"tipo_propiedad": "Tipo", "precio_usd": "Precio (USD)"},
    template="plotly_white"
)
fig_anova.show()

# Prueba Post-Hoc de Tukey HSD
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(endog=df['precio_usd'], groups=df['tipo_propiedad'], alpha=0.05)
print("\n--- Resultados Prueba de Tukey HSD ---")
print(tukey)


**Resultado ANOVA:** F = 5.12, p = 0.006

ANOVA no mira solo los promedios; mira la varianza (la dispersión). Compara qué tan separados están los promedios de los grupos entre sí (variabilidad entre grupos) contra qué tan dispersos están los precios dentro de cada misma bolsa (variabilidad dentro de los grupos).

Si la distancia entre los grupos es mucho mayor que la dispersión interna, ANOVA te dice: "Sí, el tipo de propiedad influye en el precio (p-valor < 0.05)".

El problema de ANOVA: Es una prueba global. Te dice que hay una diferencia, pero no te dice dónde. No sabés si las Casas son más caras que los Departamentos, si los PHs se diferencian de las Casas, o si los tres son completamente distintos entre sí. Para resolver esto, usamos la Prueba de Tukey.


**Pregunta → Hallazgo:** *¿El tipo de propiedad influye en el precio?* Sí, la diferencia entre grupos es estadísticamente significativa (p<0.05): las casas son en promedio ~19% más caras que los PH. Un boxplot agrupado confirmaría visualmente que la distribución de "Casa" está desplazada hacia arriba.

**Tukey  → La Prueba Post-Hoc de Tukey (HSD)**
La prueba de Tukey (Honestly Significant Difference) compara todos los pares posibles de grupos uno por uno (Casas vs. Deptos, Casas vs. PHs, Deptos vs. PHs) y corrige el margen de error estadístico para que no cometamos falsos positivos al hacer múltiples comparaciones.

**Tukey**
La interpretacion :

Vamos a interpretarlos de forma sencilla mirando la columna clave: reject (que te dice si rechazamos la idea de que son iguales) y meandiff (la diferencia de precios).

Aquí está la conclusión de tus datos:
## 1. Casa vs. Departamento (reject: True)

* Resultado: Sí hay una diferencia significativa en el precio promedio.
* Interpretación: La diferencia (meandiff) es de -$12,863.31. Como el número es negativo, significa que los Departamentos son, en promedio, unos 12,863 dólares más baratos que las Casas (o visto al revés, las Casas son significativamente más caras). El p-valor (p-adj: 0.0193) es menor a 0.05, lo que valida estadísticamente esta brecha.

## 2. Casa vs. PH (reject: True)

* Resultado: Sí hay una diferencia significativa en el precio promedio.
* Interpretación: La diferencia es de -$15,843.80. Esto nos indica que los PHs son, en promedio, unos 15,843 dólares más baratos que las Casas. Al igual que el caso anterior, el p-valor (0.0174) confirma que no es una diferencia por casualidad.

## 3. Departamento vs. PH (reject: False)

* Resultado: No hay una diferencia significativa.
* Interpretación: Aunque en tu muestra el promedio de los PHs dió unos $2,980 dólares menos que los departamentos (meandiff: -2980.48), estadísticamente se consideran similares. El p-valor es muy alto (p-adj: 0.8559), lo que significa que la dispersión de precios dentro de ambos grupos es tan grande que no podemos asegurar que un tipo sea realmente más caro que el otro en el mercado general.

------------------------------
## Resumen del Mercado
Tus datos demuestran que el mercado se divide en dos escalones claros:

   1. Las Casas están en la cima (son el grupo significativamente más caro).
   2. Los Departamentos y los PHs comparten el escalón inferior, teniendo precios promedio muy similares entre sí desde el punto de vista estadístico.



### **3.3 Categórica vs Categórica: barrio vs tipo_propiedad (Chi-cuadrado)**

In [ ]:
df.barrio.value_counts(normalize=True)

In [ ]:
df.tipo_propiedad.value_counts(normalize=True)

In [ ]:
tabla_contingencia = pd.crosstab(df["barrio"], df["tipo_propiedad"])
display(tabla_contingencia)

chi2, p_chi2, dof, ex = stats.chi2_contingency(tabla_contingencia)
print(f"Chi-cuadrado: {chi2:.2f}, p-valor: {p_chi2:.3f}")

fig_chi2 = px.histogram(
    df, x="barrio", color="tipo_propiedad", barnorm="percent", text_auto='.0f',
    title="<b>Mix Inmobiliario por Barrio (Chi² No Significativo)</b>",
    labels={"barrio": "Barrio", "percent": "Porcentaje (%)"},
    template="plotly_white"
)
fig_chi2.show()


**Pregunta → Hallazgo:** *¿El tipo de propiedad se distribuye distinto según el barrio?* No — p=0.749 (muy por encima de 0.05) nos dice que **no hay asociación significativa**: la mezcla de tipos de propiedad es prácticamente la misma en los cuatro barrios. Esto es un hallazgo tan válido como encontrar dependencia: nos permite tratar `barrio` y `tipo_propiedad` como fuentes de variación independientes al modelar precio.


***


El código realiza una **prueba de independencia de Chi-cuadrado ($\chi^2$)**, que sirve para determinar si existe una relación estadísticamente significativa entre dos variables categóricas (en este caso, `barrio` y `tipo_propiedad`).

---

### Paso a paso del código

1. **`pd.crosstab(df["barrio"], df["tipo_propiedad"])`**: Construye la **tabla de contingencia**, que es una matriz de frecuencias observadas: cuenta cuántas propiedades de cada tipo (Casa, Departamento, PH) hay en cada barrio (Centro, Costa, Norte, Sur).
2. **`stats.chi2_contingency(tabla_contingencia)`**: Ejecuta la prueba estadística y devuelve cuatro valores:
* `chi2`: El estadístico de la prueba ($\chi^2$).
* `p_chi2`: El p-valor.
* `dof`: Grados de libertad (`(filas - 1) * (columnas - 1) = (4 - 1) * (3 - 1) = 6`).
* `ex`: La matriz de frecuencias esperadas teóricas (las frecuencias que esperarías ver si las variables fueran completamente independientes).

---

### Cómo funciona la prueba de Chi-cuadrado?

La prueba evalúa dos hipótesis:

* **Hipótesis nula ($H_0$):** Las variables son **independientes** (no hay relación entre el barrio y el tipo de propiedad; la proporción de casas, departamentos y PHs se distribuye de manera similar sin importar el barrio).
* **Hipótesis alternativa ($H_1$):** Las variables son **dependientes** (el tipo de propiedad depende del barrio; ciertos barrios concentran desproporcionadamente más departamentos o casas).

El estadístico $\chi^2$ mide qué tan lejos están los valores reales observados frente a lo que se esperaría si no hubiera ninguna relación:


$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

* Si la diferencia es muy pequeña, $\chi^2$ es bajo y el p-valor es alto.
* Si la diferencia es grande, $\chi^2$ es alto y el p-valor disminuye.

---

### Interpretación de tus resultados

* **Estadístico Chi-cuadrado = 3.46**
* **p-valor = 0.749** (74.9%)

Tomando el umbral estándar de significancia ($\alpha = 0.05$):

1. **Decisión estadística:** Como el **p-valor ($0.749$) es mucho mayor que $0.05$**, **no se rechaza la hipótesis nula ($H_0$)**.
2. **Conclusión analítica:** No hay evidencia estadística para afirmar que el barrio influye en el tipo de propiedad. Ambas variables son **independientes**.
3. **Impacto en el negocio / modelo:**
* La mezcla inmobiliaria (*mix* de casas, departamentos y PHs) es prácticamente homogénea en todos los barrios analizados.
* Si estás preparando un modelo predictivo (por ejemplo, para predecir precio), saber en qué barrio está una propiedad no aporta información relevante para deducir qué tipo de propiedad es (y viceversa, no hay interacción fuerte entre ambas características).

## **4. Profundización en Métricas Estadísticas**
### **4.1 Volatilidad por Barrio**

In [ ]:
stats_barrio = df.groupby("barrio")["precio_usd"].agg(
    Media="mean", Std="std", Mediana="median"
).reset_index().sort_values(by="Media", ascending=False)
display(stats_barrio)

fig_volatilidad = px.violin(
    df, x="barrio", y="precio_usd", box=True, color="barrio",
    title="<b>Volatilidad y Riesgo por Barrio</b>",
    labels={"barrio": "Barrio", "precio_usd": "Precio (USD)"},
    category_orders={"barrio": ["Costa", "Norte", "Centro", "Sur"]},
    template="plotly_white"
)
fig_volatilidad.show()


### **4.2 Skewness, Kurtosis y Transformación Logarítmica**

In [ ]:
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Estadísticos
skew_orig = stats.skew(df["precio_usd"].dropna())
kurt_orig = stats.kurtosis(df["precio_usd"].dropna())

log_precio = np.log(df["precio_usd"].dropna())
skew_log = stats.skew(log_precio)
kurt_log = stats.kurtosis(log_precio)

# Subplots lado a lado
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"<b>Precio Original (USD)</b><br><sup>Skew: {skew_orig:.2f} | Kurt: {kurt_orig:.2f}</sup>",
        f"<b>Log(Precio USD)</b><br><sup>Skew: {skew_log:.2f} | Kurt: {kurt_log:.2f}</sup>"
    )
)

# Histograma original
fig.add_trace(
    go.Histogram(x=df["precio_usd"], nbinsx=50, name="Original", marker_color="#1f77b4"),
    row=1, col=1
)

# Histograma con transformación logarítmica
fig.add_trace(
    go.Histogram(x=log_precio, nbinsx=50, name="Log-transformado", marker_color="#2ca02c"),
    row=1, col=2
)

# Configuración de diseño
fig.update_layout(
    title_text="<b>Impacto de la Transformación Logarítmica en la Distribución</b>",
    template="plotly_white",
    showlegend=False,
    bargap=0.05
)

fig.update_xaxes(title_text="Precio USD", row=1, col=1)
fig.update_xaxes(title_text="Log(Precio USD)", row=1, col=2)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_yaxes(title_text="Frecuencia", row=1, col=2)

fig.show()

**Hallazgo:** Skewness = **2.05** (>1, asimetría positiva fuerte) y curtosis en exceso = **6.65** (>0, leptocúrtica, colas pesadas).

Traducido a decisiones de modelado: si vas a entrenar una regresión lineal sobre `precio_usd`, **conviene aplicar `log(precio_usd)`** antes de entrenar.


### **4.3 El Peligro de las Relaciones No Lineales: Antigüedad vs Precio**

In [ ]:
r_p, _ = stats.pearsonr(df["antiguedad_anios"], df["precio_usd"])
rho_s, _ = stats.spearmanr(df["antiguedad_anios"], df["precio_usd"])
tau_k, _ = stats.kendalltau(df["antiguedad_anios"], df["precio_usd"])

print(f"Pearson r: {r_p:.3f} | Spearman rho: {rho_s:.3f} | Kendall tau: {tau_k:.3f}")

fig_antiguedad = px.scatter(
    df, x="antiguedad_anios", y="precio_usd", trendline="lowess",
    trendline_color_override="red", opacity=0.4,
    title="<b>Relación No Lineal en 'U': Antigüedad vs Precio</b><br><sup>Las métricas lineales tradicionales fallan en detectar esta tendencia.</sup>",
    labels={"antiguedad_anios": "Antigüedad (Años)", "precio_usd": "Precio (USD)"},
    template="plotly_white"
)
fig_antiguedad.show()


**Hallazgo clave (y matiz importante que no está explícito en la teoría):** las tres correlaciones son **bajas** (0.10-0.15). Si te quedaras solo con estos números, concluirías que la antigüedad casi no influye en el precio. Pero mirá el promedio por tramos de edad:


Hay una **forma de U clarísima** (las propiedades muy nuevas o muy antiguas/reformadas valen más que las de mediana edad), pero ni Pearson ni Spearman ni Kendall la detectan bien, porque **ninguna de las tres captura relaciones no monótonas** (Spearman y Kendall solo mejoran sobre Pearson cuando la relación es monótona, como un crecimiento exponencial — no cuando sube y luego baja, o viceversa). Este es exactamente el caso del Tip 1 del documento, y muestra por qué "graficar antes de descartar" es una regla no negociable.


In [ ]:
bins = [0, 10, 20, 30, 40, 50, 60]
labels = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60"]

# 2. Asignar el tramo (include_lowest=True asegura que antigüedad = 0 entre en el primer bin)
df["tramo_antiguedad"] = pd.cut(
    df["antiguedad_anios"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

# 3. Agregación analítica
tabla_analitica = (
    df.groupby("tramo_antiguedad", observed=False)["precio_usd"]
    .agg(
        precio_medio="mean",
        precio_mediana="median",
        cantidad="count"
    )
    .reset_index()
)

# Formato de presentación
tabla_analitica["precio_medio_fmt"] = tabla_analitica["precio_medio"].map("USD {:,.3f}".format)
tabla_analitica["precio_mediana_fmt"] = tabla_analitica["precio_mediana"].map("USD {:,.3f}".format)

display(tabla_analitica[["tramo_antiguedad", "precio_medio_fmt", "precio_mediana_fmt", "cantidad"]])

In [ ]:
# Preparar datos agregados
resumen_grafico = (
    df.groupby("tramo_antiguedad", observed=False)["precio_usd"]
    .mean()
    .reset_index()
)

fig = go.Figure()

# Barras de precio medio
fig.add_trace(
    go.Bar(
        x=resumen_grafico["tramo_antiguedad"],
        y=resumen_grafico["precio_usd"],
        name="Precio Medio",
        marker_color="#2b5c8f",
        text=resumen_grafico["precio_usd"].map(lambda x: f"${x:,.0f}"),
        textposition="outside"
    )
)

# Línea de tendencia para enfatizar la curvatura no lineal
fig.add_trace(
    go.Scatter(
        x=resumen_grafico["tramo_antiguedad"],
        y=resumen_grafico["precio_usd"],
        mode="lines+markers",
        name="Tendencia",
        line=dict(color="#d95f02", width=3),
        marker=dict(size=8)
    )
)

fig.update_layout(
    title="<b>Evolución No Lineal: Precio Medio según Tramo de Antigüedad</b><br><sup>Explica por qué Pearson (0.13) y Spearman (0.15) no capturan la relación completa.</sup>",
    xaxis_title="Tramo de Antigüedad (años)",
    yaxis_title="Precio Medio (USD)",
    template="plotly_white",
    yaxis=dict(range=[0, resumen_grafico["precio_usd"].max() * 1.2]),
    showlegend=False
)

fig.show()

- Por qué fallaron las correlaciones clásicas: Pearson mide exclusivamente relaciones lineales y Spearman/Kendall evalúan monotonicidad estricta (si sube $X$, $Y$ siempre debe subir).

- El patrón real: Hay un valle entre los 20 y 30 años (depreciación funcional/estética) y un repunte marcado a partir de los 40-50 años (propiedades patrimoniales, estilo constructivo o ubicaciones consolidadas). Al no ser monotónica, la correlación global se cancela y se acerca a

### **4.4 Multicolinealidad**

In [ ]:
matriz_corr = df[["metros_cuadrados", "precio_usd", "impuestos_anuales_usd"]].corr().round(2)
display(matriz_corr)

fig_corr = px.imshow(
    matriz_corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="<b>Matriz de Correlación y Multicolinealidad (r = 0.98)</b>",
    template="plotly_white"
)
fig_corr.show()


**Hallazgo:** r=0.98 entre precio e impuestos — muy por encima del umbral de 0.85 del documento. Este es un caso de manual de multicolinealidad: si vas a predecir `precio_usd` con un modelo de regresión, **`impuestos_anuales_usd` es redundante y debería eliminarse** (o usarse solo una de las dos), porque prácticamente no aporta información nueva sobre el precio que no esté ya en la variable objetivo misma.


## **5. Demostración Práctica: Paradoja de Simpson**

La paradoja de Simpson es un fenómeno estadístico en el cual una tendencia que aparece en varios grupos de datos desaparece o se invierte completamente cuando los grupos se combinan.
Ocurre generalmente cuando hay una variable oculta (llamada variable de confusión) que afecta los resultados y que no se está teniendo en cuenta a simple vista.


In [ ]:
df_simpson = pd.DataFrame({
    "Agencia": ["Agencia X", "Agencia X", "Agencia Y", "Agencia Y"],
    "Segmento": ["Departamento", "Casa", "Departamento", "Casa"],
    "Tasa_Venta_Rapida": [93.1, 73.0, 86.7, 68.8]
})

fig_simpson = px.bar(
    df_simpson, x="Segmento", y="Tasa_Venta_Rapida", color="Agencia", barmode="group",
    text="Tasa_Venta_Rapida",
    title="<b>Paradoja de Simpson en el Mercado Inmobiliario</b>",
    labels={"Tasa_Venta_Rapida": "% de Ventas Rápidas (<60 días)"},
    color_discrete_sequence=["#2ca02c", "#d62728"],
    template="plotly_white"
)
fig_simpson.update_traces(texttemplate='%{text}%', textposition='outside')
fig_simpson.add_hline(y=82.6, line_dash="dot", line_color="#d62728", annotation_text="Promedio Global Y (Falso Ganador)")
fig_simpson.add_hline(y=78.0, line_dash="dot", line_color="#2ca02c", annotation_text="Promedio Global X")
fig_simpson.show()


**Pregunta → Hallazgo:** *¿Qué agencia vende más rápido?* Mirando el total, Agencia Y parece mejor (82.6% vs 78.0%).

Pero **segmentando por tipo de propiedad, Agencia X es mejor en ambos segmentos** (93.1%>86.7% en Departamento, 73.0%>68.8% en Casa). La paradoja ocurre porque Agencia Y vendió mucho más volumen de Departamentos (el segmento "fácil" de vender rápido), inflando su promedio total.

**Conclusión práctica:** nunca decidas qué agencia contratar mirando solo el total agregado — siempre desagregá por la variable oculta (aquí, el mix de tipo de propiedad que maneja cada una).


### El mecanismo del sesgo
Agencia X trabaja en el terreno difícil: el 80% de su cartera son casas ($104$ de $130$). Aunque las vende mejor que nadie ($73\%$), arrastran su promedio general hacia abajo.

Agencia Y infla su número: casi el 80% de sus operaciones son departamentos ($263$ de $340$). Su promedio no es alto porque sea más eficiente, sino porque casi no vende casas.El sesgo de agregación: el promedio global es una media ponderada.

Cuando los tamaños muestrales entre estratos están desbalanceados, el promedio total describe qué tipo de producto vende cada uno, no qué tan bien lo vende.

In [ ]:
df_detalle = pd.DataFrame({
    "Agencia": ["Agencia X", "Agencia X", "Agencia Y", "Agencia Y"],
    "Segmento": ["Departamento", "Casa", "Departamento", "Casa"],
    "Operaciones": [26, 104, 263, 77],
    "Tasa_Venta_Rapida": [93.1, 73.0, 86.7, 68.8]
})

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>1. Eficacia Real por Segmento (% Éxito)</b><br><sup>Agencia X gana en ambos</sup>",
        "<b>2. Mix de Cartera (N° Propiedades)</b><br><sup>Agencia Y tiene casi todo en el segmento fácil</sup>"
    )
)

# Panel 1: Eficacia
for ag, col in [("Agencia X", "#2ca02c"), ("Agencia Y", "#d62728")]:
    sub = df_detalle[df_detalle["Agencia"] == ag]
    fig.add_trace(
        go.Bar(name=ag, x=sub["Segmento"], y=sub["Tasa_Venta_Rapida"],
               text=sub["Tasa_Venta_Rapida"].apply(lambda v: f"{v}%"),
               textposition="outside", marker_color=col),
        row=1, col=1
    )

# Panel 2: Volumen de operaciones
for ag, col in [("Agencia X", "#2ca02c"), ("Agencia Y", "#d62728")]:
    sub = df_detalle[df_detalle["Agencia"] == ag]
    fig.add_trace(
        go.Bar(name=ag, x=sub["Segmento"], y=sub["Operaciones"],
               text=sub["Operaciones"], textposition="outside",
               marker_color=col, showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    barmode="group",
    template="plotly_white",
    title_text="<b>Desmitificando la Paradoja de Simpson</b>",
    yaxis1=dict(range=[0, 110], title="% Venta Rápida"),
    yaxis2=dict(title="Cantidad de Propiedades")
)
fig.show()

In [ ]:
df_detalle = pd.DataFrame({
    "Agencia": ["Agencia X", "Agencia X", "Agencia Y", "Agencia Y"],
    "Segmento": ["Departamento", "Casa", "Departamento", "Casa"],
    "Operaciones": [26, 104, 263, 77],
    "Tasa_Venta_Rapida": [93.1, 73.0, 86.7, 68.8]
})

# 1. Calcular el número absoluto de ventas rápidas
df_detalle["Ventas_Rapidas"] = (df_detalle["Operaciones"] * (df_detalle["Tasa_Venta_Rapida"] / 100)).round()

# 2. Calcular los totales agregados por agencia
totales = df_detalle.groupby("Agencia")[["Operaciones", "Ventas_Rapidas"]].sum().reset_index()
totales["Segmento"] = "TOTAL (Agregado)"
totales["Tasa_Venta_Rapida"] = (totales["Ventas_Rapidas"] / totales["Operaciones"]) * 100

# 3. Unir segmentos y total
df_completo = pd.concat([df_detalle, totales], ignore_index=True)

# 4. Graficar
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>Tasa de Éxito (%)</b><br><sup>X gana en Depto y Casa, pero Y gana en el Total</sup>",
        "<b>Volumen de Operaciones</b><br><sup>Explicación: Y concentró casi todo en Depto</sup>"
    )
)

colores = {"Agencia X": "#2ca02c", "Agencia Y": "#d62728"}

for ag in ["Agencia X", "Agencia Y"]:
    sub = df_completo[df_completo["Agencia"] == ag]

    # Panel 1: Tasas
    fig.add_trace(
        go.Bar(name=ag, x=sub["Segmento"], y=sub["Tasa_Venta_Rapida"],
               text=sub["Tasa_Venta_Rapida"].apply(lambda v: f"{v:.1f}%"),
               textposition="outside", marker_color=colores[ag]),
        row=1, col=1
    )

    # Panel 2: Volúmenes
    fig.add_trace(
        go.Bar(name=ag, x=sub["Segmento"], y=sub["Operaciones"],
               text=sub["Operaciones"], textposition="outside",
               marker_color=colores[ag], showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    barmode="group",
    template="plotly_white",
    yaxis1=dict(range=[0, 110], title="% Ventas Rápidas"),
    yaxis2=dict(title="Cantidad Total"),
    title_text="<b>Paradoja de Simpson: Detalle por Segmento vs Total Agregado</b>"
)
fig.show()

## Resumen de hallazgos

1. `precio_usd` es asimétrico y leptocúrtico → requiere transformación log antes de modelar.
2. El tamaño (m²) explica el precio moderadamente (r=0.64); el tipo de propiedad también influye (ANOVA significativo); barrio y tipo son independientes entre sí.
3. Costa es el barrio más caro y más volátil; Sur el más económico y estable.
4. La antigüedad **no tiene correlación lineal ni monótona** con el precio, pero sí una relación en U real — lección sobre las limitaciones de los tres coeficientes de correlación.
5. Impuestos y precio están multicolineales (r=0.98) — candidato a eliminar en un modelo predictivo.
6. La paradoja de Simpson recuerda que un ranking agregado puede mentir si no se desagrega por la variable de confusión.
